# MLP Permutation Importance

This notebook loads the saved MLP default-risk model and estimates permutation importance on the held-out test split. It complements the SHAP notebook by asking how much model performance drops when a feature, or a group of related transformed features, is shuffled.

Positive importance means shuffling that feature made the model worse. Values near zero mean the model could mostly recover from losing that feature, or the feature is redundant with other inputs.

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

try:
    from credit_data import add_features
except ModuleNotFoundError:
    import sys

    sys.path.append(str(Path.cwd() / "src"))
    from credit_data import add_features

DEFAULT_CATEGORICAL_COLS = ["SEX", "EDUCATION", "MARRIAGE", "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
PAY_STATUS_COLS = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
DELAY_SUMMARY_COLS = ["MAX_PAY_DELAY", "AVG_PAY_DELAY", "MONTHS_WITH_DELAY"]
FOCUSED_PERMUTATION_FEATURES = ["SEX", *PAY_STATUS_COLS, *DELAY_SUMMARY_COLS]


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def load_credit_default_data():
    data_paths = [
        Path("../data/default_of_credit_card_clients.xls"),
        Path("data/default_of_credit_card_clients.xls"),
    ]
    data_path = next(path for path in data_paths if path.exists())
    return pd.read_excel(data_path, header=1), data_path, "default payment next month"


def load_torch_artifact(device=None):
    model_paths = [
        Path("../models/credit_default_mlp.pt"),
        Path("models/credit_default_mlp.pt"),
    ]
    model_path = next(path for path in model_paths if path.exists())
    try:
        artifact = torch.load(model_path, map_location=device, weights_only=False)
    except TypeError:
        artifact = torch.load(model_path, map_location=device)
    return artifact, model_path


def prepare_feature_set(source_df, target_col, use_feature_engineering, seed, categorical_cols=DEFAULT_CATEGORICAL_COLS, y_dtype=None, output_dtype=None):
    model_df = add_features(source_df) if use_feature_engineering else source_df.copy()
    X_raw = model_df.drop(columns=["ID", target_col])
    y_raw = model_df[target_col].astype(y_dtype) if y_dtype is not None else model_df[target_col]
    numeric_cols = [col for col in X_raw.columns if col not in categorical_cols]

    X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
        X_raw, y_raw, test_size=0.30, random_state=seed, stratify=y_raw
    )
    X_val_raw, X_test_raw, y_val, y_test = train_test_split(
        X_temp_raw, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
    )

    preprocess = ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), numeric_cols),
            ("categorical", make_one_hot_encoder(), categorical_cols),
        ]
    )

    X_train = preprocess.fit_transform(X_train_raw)
    X_val = preprocess.transform(X_val_raw)
    X_test = preprocess.transform(X_test_raw)

    if output_dtype is not None:
        X_train = X_train.astype(output_dtype)
        X_val = X_val.astype(output_dtype)
        X_test = X_test.astype(output_dtype)

    return {
        "preprocess": preprocess,
        "categorical_cols": list(categorical_cols),
        "numeric_cols": numeric_cols,
        "X_train_raw": X_train_raw,
        "X_val_raw": X_val_raw,
        "X_test_raw": X_test_raw,
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
    }


class CreditDefaultMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)


def make_credit_default_mlp(input_dim, device=None):
    model = CreditDefaultMLP(input_dim)
    return model.to(device) if device is not None else model


def make_torch_dataset(X_array, y_series=None):
    X_tensor = torch.tensor(X_array, dtype=torch.float32)
    if y_series is None:
        return TensorDataset(X_tensor)
    y_values = y_series.to_numpy() if hasattr(y_series, "to_numpy") else y_series
    y_tensor = torch.tensor(y_values, dtype=torch.float32).view(-1, 1)
    return TensorDataset(X_tensor, y_tensor)


def predict_mlp_proba(model, X_array, y_series=None, device=None, batch_size=1024):
    loader = DataLoader(make_torch_dataset(X_array, y_series), batch_size=batch_size, shuffle=False)
    model.eval()
    probs = []
    targets = []
    with torch.no_grad():
        for batch in loader:
            if y_series is None:
                (xb,) = batch
            else:
                xb, yb = batch
                targets.append(yb.numpy().ravel())
            logits = model(xb.to(device) if device is not None else xb)
            probs.append(torch.sigmoid(logits).cpu().numpy().ravel())
    y_prob = np.concatenate(probs)
    if y_series is None:
        return y_prob
    return np.concatenate(targets), y_prob


def make_mlp_predict_proba(model, device=None, batch_size=1024):
    def predict_proba(X_array):
        return predict_mlp_proba(model, X_array, device=device, batch_size=batch_size)
    return predict_proba


def transformed_feature_names(column_transformer):
    names = []
    for transformer_name, transformer, columns in column_transformer.transformers_:
        if transformer_name == "remainder" and transformer == "drop":
            continue
        if transformer_name == "categorical":
            names.extend(transformer.get_feature_names_out(columns).tolist())
        else:
            names.extend(list(columns))
    return names


def original_feature_name(transformed_name, categorical_cols=DEFAULT_CATEGORICAL_COLS):
    for col in categorical_cols:
        if transformed_name.startswith(f"{col}_"):
            return col
    return transformed_name


def feature_groups_from_names(feature_names, categorical_cols=DEFAULT_CATEGORICAL_COLS):
    groups = {}
    for idx, feature_name in enumerate(feature_names):
        groups.setdefault(original_feature_name(feature_name, categorical_cols), []).append(idx)
    return groups


def score_binary_predictions(y_true, y_prob):
    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
    }


def sample_rows(X, y, sample_size, random_state):
    sample_size = min(sample_size, X.shape[0])
    rng = np.random.default_rng(random_state)
    sample_idx = rng.choice(X.shape[0], size=sample_size, replace=False)
    y_sample = y.iloc[sample_idx].to_numpy() if hasattr(y, "iloc") else y[sample_idx]
    return X[sample_idx], y_sample, sample_idx


def single_column_groups(feature_names):
    return [[idx] for idx in range(len(feature_names))], list(feature_names)


def grouped_feature_columns(feature_groups):
    group_names = list(feature_groups.keys())
    group_cols = [feature_groups[name] for name in group_names]
    return group_cols, group_names


def delay_permutation_groups(feature_groups):
    repayment_status_cols = sorted(
        col_idx for feature in PAY_STATUS_COLS for col_idx in feature_groups[feature]
    )
    custom_groups = [repayment_status_cols]
    custom_group_names = ["ALL_PAY_STATUS"]

    if all(feature in feature_groups for feature in DELAY_SUMMARY_COLS):
        engineered_delay_cols = sorted(
            col_idx for feature in DELAY_SUMMARY_COLS for col_idx in feature_groups[feature]
        )
        custom_groups.append(engineered_delay_cols)
        custom_group_names.append("ENGINEERED_DELAY_SUMMARIES")
        custom_groups.append(sorted(set(repayment_status_cols + engineered_delay_cols)))
        custom_group_names.append("ALL_DELAY_FEATURES")

    return custom_groups, custom_group_names


def calculate_permutation_importance(X, y, predict_proba, groups, group_names, metric_name="roc_auc", n_repeats=5, random_state=42):
    rng = np.random.default_rng(random_state)
    baseline_prob = predict_proba(X)
    baseline_metric = score_binary_predictions(y, baseline_prob)[metric_name]
    rows = []

    for group_name, group_cols in zip(group_names, groups):
        permuted_metrics = []
        for _ in range(n_repeats):
            X_permuted = X.copy()
            shuffled_rows = rng.permutation(X.shape[0])
            X_permuted[:, group_cols] = X_permuted[shuffled_rows][:, group_cols]
            permuted_prob = predict_proba(X_permuted)
            permuted_metrics.append(score_binary_predictions(y, permuted_prob)[metric_name])

        permuted_metrics = np.asarray(permuted_metrics)
        drops = baseline_metric - permuted_metrics
        rows.append({
            "feature": group_name,
            "n_transformed_columns": len(group_cols),
            "baseline_score": baseline_metric,
            "permuted_score_mean": permuted_metrics.mean(),
            "importance_mean": drops.mean(),
            "importance_std": drops.std(ddof=1) if n_repeats > 1 else 0.0,
        })

    return pd.DataFrame(rows).sort_values("importance_mean", ascending=False).reset_index(drop=True)


In [ ]:
df, DATA_PATH, target_col = load_credit_default_data()
artifact, MODEL_PATH = load_torch_artifact(device=device)

print("Data:", DATA_PATH, df.shape)
print("Model:", MODEL_PATH)
print("Saved feature set:", artifact.get("feature_set"))
print("Saved threshold:", round(float(artifact.get("best_threshold", 0.5)), 3))


## Rebuild the Training Preprocessing

The model artifact stores feature-set metadata, but not the fitted `ColumnTransformer`. This recreates the same split, scaling, one-hot encoding, and optional engineered features used during MLP training.

In [ ]:
categorical_cols = artifact.get("categorical_cols", DEFAULT_CATEGORICAL_COLS)

active_data = prepare_feature_set(
    df,
    target_col=target_col,
    use_feature_engineering=artifact.get("use_feature_engineering", True),
    seed=SEED,
    categorical_cols=categorical_cols,
    y_dtype=np.float32,
    output_dtype=np.float32,
)
X_train = active_data["X_train"]
X_val = active_data["X_val"]
X_test = active_data["X_test"]
y_val = active_data["y_val"]
y_test = active_data["y_test"]
preprocess = active_data["preprocess"]

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Input dim matches artifact:", X_train.shape[1] == artifact["input_dim"])


In [ ]:
model = make_credit_default_mlp(input_dim=artifact["input_dim"], device=device)
model.load_state_dict(artifact["model_state_dict"])
model.eval()

print(model)


In [ ]:
predict_proba = make_mlp_predict_proba(model, device=device)

y_test_prob = predict_proba(X_test)
baseline_scores = score_binary_predictions(y_test, y_test_prob)

print("Baseline test ROC-AUC:", round(baseline_scores["roc_auc"], 4))
print("Baseline test PR-AUC:", round(baseline_scores["pr_auc"], 4))


## Feature Names and Groups

The MLP sees transformed columns. Numeric columns stay as one transformed feature each, while categorical columns such as `PAY_0` and `SEX` become one-hot indicators like `PAY_0_2` or `SEX_1`.

The grouped importance section below shuffles all one-hot columns from the same original feature together.

In [ ]:
feature_names = transformed_feature_names(preprocess)
feature_groups = feature_groups_from_names(feature_names, categorical_cols)

print("Transformed features:", len(feature_names))
print("Original/grouped features:", len(feature_groups))
print("PAY_0 transformed columns:", [feature_names[i] for i in feature_groups["PAY_0"]])
print("SEX transformed columns:", [feature_names[i] for i in feature_groups["SEX"]])


## Permutation Importance Helpers

`metric_name` controls which performance metric is used. ROC-AUC is the default because it is threshold-free. PR-AUC is often useful too because defaults are the minority class.

In [ ]:
# Use a sample to keep the notebook responsive. Increase this or set it to X_test.shape[0] for a fuller run.
PERMUTATION_SAMPLE_SIZE = min(2000, X_test.shape[0])
N_REPEATS = 5
METRIC_NAME = "roc_auc"

X_perm, y_perm, sample_idx = sample_rows(
    X_test,
    y_test,
    sample_size=PERMUTATION_SAMPLE_SIZE,
    random_state=SEED,
)

print("Permutation sample:", X_perm.shape)
print("Metric:", METRIC_NAME)
print("Repeats:", N_REPEATS)


## Transformed-Feature Importance

This treats every model input column separately. It is closest to the exact representation seen by the MLP, but one-hot categorical features are split into separate columns.

In [ ]:
transformed_groups, transformed_group_names = single_column_groups(feature_names)

transformed_importance = calculate_permutation_importance(
    X_perm,
    y_perm,
    predict_proba=predict_proba,
    groups=transformed_groups,
    group_names=transformed_group_names,
    metric_name=METRIC_NAME,
    n_repeats=N_REPEATS,
    random_state=SEED,
)

transformed_importance.head(25)


In [ ]:
top_transformed = transformed_importance.head(25).iloc[::-1]

plt.figure(figsize=(8, 7))
plt.barh(top_transformed["feature"], top_transformed["importance_mean"], xerr=top_transformed["importance_std"])
plt.axvline(0, color="black", linewidth=1)
plt.title(f"Top transformed features by permutation importance ({METRIC_NAME})")
plt.xlabel("Baseline score - permuted score")
plt.tight_layout()
plt.show()

## Grouped Original-Feature Importance

This is usually easier to interpret for categorical variables. For example, all `PAY_0_*` one-hot columns are shuffled together, so the result answers how much the model depends on the original `PAY_0` repayment-status field.

In [ ]:
group_cols, group_names = grouped_feature_columns(feature_groups)

grouped_importance = calculate_permutation_importance(
    X_perm,
    y_perm,
    predict_proba=predict_proba,
    groups=group_cols,
    group_names=group_names,
    metric_name=METRIC_NAME,
    n_repeats=N_REPEATS,
    random_state=SEED,
)

grouped_importance.head(25)


In [ ]:
top_grouped = grouped_importance.head(25).iloc[::-1]

plt.figure(figsize=(8, 7))
plt.barh(top_grouped["feature"], top_grouped["importance_mean"], xerr=top_grouped["importance_std"])
plt.axvline(0, color="black", linewidth=1)
plt.title(f"Top original features by grouped permutation importance ({METRIC_NAME})")
plt.xlabel("Baseline score - permuted score")
plt.tight_layout()
plt.show()

## Sensitive and Payment-Status Checks

`SEX` is included in the dataset and is one-hot encoded. These focused tables make it easy to see whether shuffling sensitive demographic information or payment-status fields changes held-out performance.

In [ ]:
grouped_importance[
    grouped_importance["feature"].isin(FOCUSED_PERMUTATION_FEATURES)
].sort_values("importance_mean", ascending=False)


## Optional: Shuffle All Repayment-Status Features Together

Individual permutation can understate importance when related features carry overlapping information. This optional check shuffles all monthly repayment-status fields together.

In [ ]:
custom_groups, custom_group_names = delay_permutation_groups(feature_groups)

custom_importance = calculate_permutation_importance(
    X_perm,
    y_perm,
    predict_proba=predict_proba,
    groups=custom_groups,
    group_names=custom_group_names,
    metric_name=METRIC_NAME,
    n_repeats=N_REPEATS,
    random_state=SEED,
)

custom_importance
